[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [LightGBM and XGBoost, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/lightgbm-and-xgboost-deep-dive.html)

# Regularization and Sampling


## What you will be able to do

Limit how closely each LightGBM or XGBoost tree fits the training rows, with a penalty on its leaves'
values, a minimum size for a leaf, and a random sample of the rows or columns for every tree, and
tell a setting's effect from the noise a seed adds.

## The idea

### The problem

The **Tree Shape** notebook limited a tree's size, and a tree of the right size can still fit noise.
A leaf that holds the days of only a few stations takes their average as its value, however unusual
those days were, and every later tree builds on that answer. In 15,000 training rows some small
groups look special by chance, and a model that trusts each of them learns things the test
stations will not repeat.

Both libraries can hold a tree back with a penalty, a minimum leaf size and sampling, and their
defaults differ. XGBoost penalizes leaf values unless told otherwise, and LightGBM does not. LightGBM
refuses a leaf of fewer than 20 rows, XGBoost's `min_child_weight` of 1 lets a squared-error leaf hold
a single row, and neither library samples rows or columns unless asked.

### A formal definition

> **Regularization** in gradient boosting is any limit on how closely each tree may fit the training
> rows. A **penalty** shrinks a leaf's value toward zero: `reg_lambda` adds λ/2 times the sum of the
> squared values to the loss, and `reg_alpha` adds α times the sum of their sizes.
> A **minimum leaf size** refuses a split that would leave a leaf with too little in it. **Sampling** fits each tree on a random share of the rows, `subsample`,
> or of the columns, `colsample_bytree`.

### Why it works that way

For squared error, a leaf's value under a penalty λ is the sum of its residuals divided by the number
of its rows plus λ. A leaf of 3,000 rows hardly notices a λ of 1, and a leaf of three rows has its
value cut by a quarter, so the penalty falls hardest on the small groups most likely to be noise. λ
also enters the gain a split is judged by, so a split that gains little is not made at all.

A minimum leaf size says the same thing more bluntly, by refusing small leaves outright. LightGBM's
`min_child_samples` counts rows. XGBoost's `min_child_weight` adds up a weight per row, the second
derivative of the loss, which is 1 for squared error, at most a quarter for a row of a two-class
classifier and at most a half for a row of XGBoost's classifier of three or more classes, so the two
settings mean the same only for squared error.

Sampling works differently. Each tree sees another part of the table, so the trees disagree a little,
and their sum averages over the disagreement. It brings randomness with it, so a result depends on the
seed. LightGBM samples rows only when `subsample_freq` is above 0, which by default it is not.

### Where this shows up

The **Tuning** notebook searches over minimum leaf sizes and sampling shares with cross-validation.
They matter most on a small or noisy table, where a leaf of a few rows is likeliest to fit noise.

### What this notebook covers

- a penalty's effect on one leaf, worked out by hand
- penalties, minimum leaf sizes and sampling against the validation loss
- how much the seed alone moves a sampled model
- three mistakes, two of which raise nothing

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read it,
and read the output underneath it. Everything from Setup onward is where you start running things.

```python
import numpy as np
import xgboost as xgb

x = np.array([[0], [0], [1], [1], [1]])
y = np.array([1.0, 3.0, 10.0, 12.0, 14.0])
for penalty in (0, 1, 10):
    stump = xgb.XGBRegressor(n_estimators=1, max_depth=1, learning_rate=1.0, reg_lambda=penalty,
                             base_score=0).fit(x, y)
    print(penalty, stump.predict(np.array([[0], [1]])).round(2))
```

```
0 [ 2. 12.]
1 [1.33 9.  ]
10 [2.67 2.67]
```

Each model is one split, with its full step added. With no penalty the two leaves predict their
groups' means, 2 and 12. With λ = 1 each leaf's sum is divided by its rows plus one, 4 ÷ 3 and 36 ÷ 4.
With λ = 10 the split no longer gains enough to be made, and the one leaf left predicts 40 ÷ 15 for
every row.

## Setup

The table, the two boosting libraries, and a score.

- `numpy` and `pandas` build the table
- `lightgbm` and `xgboost` are LightGBM and XGBoost, and the cell prints their versions
- `mean_squared_error` scores a model's predictions

`station_days` makes the same 20,000 made-up station-days as the **Why Gradient Boosting** notebook,
drawn from a generator started from a fixed seed, so the table is the same on every run. The cell
splits the stations into 300 for training, 50 for validation and 50 for testing, keeping all of a
station's days on one side, and then takes each part's eight features as `X_train`, `X_valid` and
`X_test` and its lowest temperatures as `y_train`, `y_valid` and `y_test`.

In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from sklearn.metrics import mean_squared_error

TERRAINS = ["coastal", "fjord", "inland", "valley", "mountain"]


def station_days(n_stations=400, days=50, seed=2026):
    """Made-up daily minimum temperatures at weather stations in Norway, one row per station and day."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n_stations)
    coast_km = np.minimum(rng.exponential(30, n_stations), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n_stations) + 2.5 * coast_km, 0, 1600)
    terrain = rng.choice(TERRAINS, n_stations, p=[0.2, 0.2, 0.25, 0.2, 0.15])
    has_snow_sensor = rng.uniform(size=n_stations) < 0.7
    offset = rng.normal(0, 0.8, n_stations)
    station = np.repeat(np.arange(n_stations), days)
    n = n_stations * days
    day = rng.integers(1, 366, n)
    cloud = np.clip(rng.beta(1.2, 1.0, n), 0, 1)
    wind = rng.gamma(2.0, 2.0, n)
    season = np.cos(2 * np.pi * (day - 20) / 365)
    lat, elev, coast = latitude[station], elevation[station], coast_km[station]
    snow = np.where(season > 0.2, np.clip(rng.normal(30 * season + 0.03 * elev + 2 * (lat - 58), 15), 0, None), 0.0)
    shelter = pd.Series(terrain[station]).map({"coastal": 0.4, "fjord": 1.0, "inland": 1.2, "valley": 1.7, "mountain": 0.8}).to_numpy()
    mean = 8 - 0.55 * (lat - 58) - 0.006 * elev - 9 * season + 3.5 * np.exp(-coast / 40) * season
    cooling = 7 * (1 - cloud) * np.exp(-wind / 3) * (0.4 + (season + 1) / 2) * shelter
    tmin = mean - 2.5 - cooling - 2.0 * (snow > 10) * (1 - cloud) + offset[station] + rng.normal(0, 1.2, n)
    snow_cm = np.where(has_snow_sensor[station], snow.round(0), np.nan)
    return pd.DataFrame({
        "station": [f"ST{s + 1:04d}" for s in station],
        "latitude": lat.round(2), "elevation_m": elev.round(0).astype(int), "coast_km": coast.round(1),
        "terrain": pd.Categorical(terrain[station], categories=TERRAINS),
        "day_of_year": day, "cloud_cover": cloud.round(2), "wind_ms": wind.round(1), "snow_cm": snow_cm,
        "tmin": tmin.round(1),
    })


FEATURES = ["latitude", "elevation_m", "coast_km", "terrain", "day_of_year", "cloud_cover", "wind_ms", "snow_cm"]
days = station_days()
stations = days["station"].unique()
order = np.random.default_rng(0).permutation(len(stations))
train = days[days["station"].isin(stations[order[:300]])]
valid = days[days["station"].isin(stations[order[300:350]])]
test = days[days["station"].isin(stations[order[350:]])]
X_train, y_train = train[FEATURES], train["tmin"]
X_valid, y_valid = valid[FEATURES], valid["tmin"]
X_test, y_test = test[FEATURES], test["tmin"]

print("lightgbm", lgb.__version__, "| xgboost", xgb.__version__, "|", len(train), "training rows,",
      len(valid), "validation rows,", len(test), "test rows")

lightgbm 4.6.0 | xgboost 3.4.1 | 15000 training rows, 2500 validation rows, 2500 test rows


## Worked examples

### A penalty on one leaf

The cell below fits a single XGBoost split on the stations three times, with its full step added and a
starting prediction of 0, so that each leaf's value can be checked by hand.

In [2]:
for penalty in (0, 1, 1000):
    stump = xgb.XGBRegressor(n_estimators=1, max_depth=1, learning_rate=1.0, reg_lambda=penalty,
                             base_score=0).fit(X_train, y_train)
    nodes = stump.get_booster().trees_to_dataframe()
    leaves = nodes[nodes["Feature"] == "Leaf"]
    print(f"reg_lambda {penalty:4}: split on {nodes.loc[0, 'Feature']} at {nodes.loc[0, 'Split']}, "
          f"leaf values {leaves['Gain'].round(3).tolist()}, rows {leaves['Cover'].astype(int).tolist()}")

reg_lambda    0: split on day_of_year at 94.0, leaf values [-8.164, 0.726], rows [3794, 11206]
reg_lambda    1: split on day_of_year at 94.0, leaf values [-8.162, 0.726], rows [3794, 11206]
reg_lambda 1000: split on snow_cm at 7.0, leaf values [0.855, -6.152], rows [10667, 4333]


With no penalty the leaf for the 3,794 days before day 94 holds −8.164, and λ = 1 moves it to
−8.162. With λ = 1000 the stump splits somewhere else, on snow, because the penalty enters the gain
of every candidate split and changes which one wins.

In [3]:
early = X_train["day_of_year"] < 94
print("rows before day 94:    ", early.sum())
print("mean of the early days:", round(y_train[early].mean(), 3))
print("sum over rows + 1:     ", round(y_train[early].sum() / (early.sum() + 1), 3))
print("sum over rows + 1000:  ", round(y_train[early].sum() / (early.sum() + 1000), 3))

rows before day 94:     3794
mean of the early days: -8.164
sum over rows + 1:      -8.162
sum over rows + 1000:   -6.461


The early days' mean is −8.164, and their sum over their rows plus 1 is −8.162, the leaf λ = 1 gave.
Over their rows plus 1000 it would be −6.461. With 3,794 rows in the leaf, a λ of 1 changes its value
by about 0.002, and a λ of 1000 would cut it by about a fifth.

### Penalties against the validation loss

The helpers below train a model with the settings they are given, stop it early on the validation
stations at a learning rate of 0.1, and return its number of trees and its validation loss.

In [4]:
def lightgbm_validation(**settings):
    model = lgb.LGBMRegressor(n_estimators=3000, learning_rate=0.1, random_state=0, verbose=-1, **settings)
    model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], callbacks=[lgb.early_stopping(100, verbose=False)])
    return model.best_iteration_, round(float(model.best_score_["valid_0"]["l2"]), 3)


def xgboost_validation(**settings):
    model = xgb.XGBRegressor(n_estimators=3000, learning_rate=0.1, early_stopping_rounds=100, random_state=0,
                             **settings)
    model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], verbose=False)
    return model.best_iteration + 1, round(mean_squared_error(y_valid, model.predict(X_valid)), 3)


for penalty in (0, 1, 10, 100):
    print(f"reg_lambda {penalty:3}: LightGBM {lightgbm_validation(reg_lambda=penalty)}, "
          f"XGBoost {xgboost_validation(reg_lambda=penalty)}")

reg_lambda   0: LightGBM (120, 2.546), XGBoost (154, 2.533)
reg_lambda   1: LightGBM (146, 2.557), XGBoost (77, 2.526)
reg_lambda  10: LightGBM (347, 2.498), XGBoost (142, 2.556)
reg_lambda 100: LightGBM (329, 2.485), XGBoost (454, 2.572)


Each pair is a number of trees and a validation loss. LightGBM's loss is lower with a penalty of 10 or
100, 2.498 and 2.485, than with none, its default, at 2.546. XGBoost's is lowest at its default of 1,
2.526, and highest at 100, 2.572. With a penalty of 100 both libraries stop later than with none,
after 329 and 454 trees against 120 and 154, since every tree's steps are smaller. Most of these
differences are a few hundredths. None of these models samples rows or columns, so the seed has
nothing to change in them, and what could reorder them is a different choice of the 50 validation
stations.

### Minimum leaf sizes

LightGBM's `min_child_samples` sets the fewest rows a leaf may hold. XGBoost's `min_child_weight` sets
the smallest sum of the loss's second derivatives in a leaf, and for squared error each row adds 1 to
that sum, so a leaf's weight is its number of rows. The first example printed that weight, XGBoost's
`Cover`, as rows, and the hand check counted the same 3,794.

In [5]:
for rows in (5, 20, 100, 400):
    print(f"min_child_samples {rows:3}: LightGBM {lightgbm_validation(min_child_samples=rows)}")
for weight in (1, 10, 100):
    print(f"min_child_weight {weight:3}: XGBoost {xgboost_validation(min_child_weight=weight)}")

min_child_samples   5: LightGBM (86, 2.587)
min_child_samples  20: LightGBM (120, 2.546)
min_child_samples 100: LightGBM (165, 2.455)
min_child_samples 400: LightGBM (171, 2.581)
min_child_weight   1: XGBoost (77, 2.526)
min_child_weight  10: XGBoost (97, 2.572)
min_child_weight 100: XGBoost (357, 2.518)


For LightGBM, a minimum of 100 rows gives the lowest validation loss, 2.455, and both 5 and 400 do
worse than the default of 20. For XGBoost the three weights end within 0.06 of each other, and the
loss neither rises nor falls steadily as the weight grows.

### Sampling rows and columns for every tree

LightGBM samples rows only when `subsample_freq` is above 0, and 1 draws a new sample for every tree.

In [6]:
for share in (1.0, 0.7, 0.5):
    print(f"subsample {share}: LightGBM {lightgbm_validation(subsample=share, subsample_freq=1)}, "
          f"XGBoost {xgboost_validation(subsample=share)}")
for share in (1.0, 0.5):
    print(f"colsample_bytree {share}: LightGBM {lightgbm_validation(colsample_bytree=share)}, "
          f"XGBoost {xgboost_validation(colsample_bytree=share)}")

subsample 1.0: LightGBM (120, 2.546), XGBoost (77, 2.526)
subsample 0.7: LightGBM (280, 2.549), XGBoost (151, 2.474)
subsample 0.5: LightGBM (120, 2.543), XGBoost (62, 2.542)
colsample_bytree 1.0: LightGBM (120, 2.546), XGBoost (77, 2.526)
colsample_bytree 0.5: LightGBM (272, 2.73), XGBoost (254, 2.611)


From seed 0, sampling the rows moves LightGBM's validation loss by less than 0.01 and XGBoost's by
about 0.05 at most, in both directions, and the last worked example shows how much further another
seed moves it. Sampling half the columns makes both worse, 2.73 and 2.611. With eight features, each
tree is given four of them to split on, and on these stations that loses more than the variety between
trees gains.

### How much the seed alone moves a sampled model

The cell below trains one LightGBM model three times, from three seeds, sampling half the rows for
every tree.

In [7]:
for seed in (0, 1, 2):
    model = lgb.LGBMRegressor(n_estimators=3000, learning_rate=0.1, subsample=0.5, subsample_freq=1,
                              random_state=seed, verbose=-1)
    model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], callbacks=[lgb.early_stopping(100, verbose=False)])
    print(f"seed {seed}: best {model.best_iteration_} trees, validation {model.best_score_['valid_0']['l2']:.3f}")

seed 0: best 120 trees, validation 2.543
seed 1: best 217 trees, validation 2.538
seed 2: best 157 trees, validation 2.464


The same settings give 2.543, 2.538 and 2.464 from three seeds, a spread of almost 0.08, and stop
after 120, 217 and 157 trees, so one seed cannot rank the sampling settings above. The penalties and
leaf sizes sample nothing, and their differences depend instead on which 50 stations are held out for
validation. The **Tuning** notebook measures that with cross-validation before it compares settings.

## Your turn

Five tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/lightgbm-and-xgboost-deep-dive/07-regularization-and-sampling-solutions.ipynb).

**1.** Train XGBoost with `reg_alpha=10`, the penalty on the size of each leaf's value, with `xgboost_validation`, and print its trees and validation loss beside the default's.

In [8]:
# your code here

**2.** Fit the XGBoost stump of the first worked example with `reg_lambda=100`, and check the value of its first leaf against the sum of that leaf's answers over its rows plus 100.

In [9]:
# your code here

**3.** Train LightGBM with `subsample=0.7` and `subsample_freq=1` from seeds 0, 1 and 2, and print the lowest and highest validation loss of the three.

In [10]:
# your code here

**4.** Train both libraries with `colsample_bytree=0.75`, and print their trees and validation losses.

In [11]:
# your code here

**5.** Train LightGBM with `num_leaves=8`, `min_child_samples=100` and `reg_lambda=10` together, and print its trees and validation loss.

In [12]:
# your code here

## Common errors

### No error, and a `subsample` that LightGBM ignores

In [13]:
plain = lgb.LGBMRegressor(n_estimators=200, random_state=0, verbose=-1).fit(X_train, y_train)
half = lgb.LGBMRegressor(n_estimators=200, subsample=0.5, random_state=0, verbose=-1).fit(X_train, y_train)
sampled = lgb.LGBMRegressor(n_estimators=200, subsample=0.5, subsample_freq=1, random_state=0, verbose=-1)
sampled.fit(X_train, y_train)

print("subsample=0.5 alone, same predictions as none:  ", np.array_equal(plain.predict(X_valid), half.predict(X_valid)))
print("with subsample_freq=1, same predictions as none:", np.array_equal(plain.predict(X_valid), sampled.predict(X_valid)))

subsample=0.5 alone, same predictions as none:   True
with subsample_freq=1, same predictions as none: False


Without `subsample_freq`, LightGBM keeps its default of 0, which means it never draws a sample, so
`subsample=0.5` changes nothing and the predictions are identical to a model without it. Set
`subsample_freq=1` to draw a new sample for every tree.

### `XGBoostError: value 50 for Parameter subsample exceed bound [0,1]`

A share written as a percentage, 50 where 0.5 was meant, stops XGBoost before it trains.

In [14]:
xgb.XGBRegressor(n_estimators=200, subsample=50).fit(X_train, y_train)

XGBoostError: value 50 for Parameter subsample exceed bound [0,1]
subsample: Row subsample ratio of training instance.

XGBoost names the setting as it was given and the bound it must stay within, 0 to 1, and a second
line describes the setting. A share is a fraction of the rows, so write 0.5.

### No error, and a model that predicts one value for every station

In [15]:
timid = lgb.LGBMRegressor(n_estimators=100, min_child_samples=8000, random_state=0, verbose=-1).fit(X_train, y_train)
predictions = timid.predict(X_valid)
print("trees:", timid.booster_.num_trees(), "| different predictions:", len(np.unique(predictions)))
print("prediction:", round(float(predictions[0]), 3), "| training mean:", round(float(y_train.mean()), 3))

trees: 1 | different predictions: 1
prediction: -1.522 | training mean: -1.522


A leaf must hold 8,000 rows, more than half of the 15,000, so no split is possible, and the model is
one leaf that predicts the training mean, −1.522, for every station. LightGBM kept one tree, since no
later tree could split either. It tried all 100 rounds, and with `verbose=-1` it said nothing about any
of them. Keep a minimum leaf size small next to the number of training rows, as the worked examples'
5 to 400 are.

## Recap

- You can shrink leaf values with `reg_lambda` and `reg_alpha`, and check a leaf's value by hand.
- You can refuse small leaves with `min_child_samples` in LightGBM and `min_child_weight` in XGBoost.
- You can fit each tree on a sample of the rows or columns, with `subsample_freq` set in LightGBM.
- You can run a sampled model from several seeds before reading anything into a small difference.

## What is next

The **Missing Values** notebook looks at the gaps in the table, such as the snow readings at stations
with no sensor, and how each split decides where they go.

---

&#8592; **Previous:** [Tree Shape](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/lightgbm-and-xgboost-deep-dive/06-tree-shape.ipynb)  &nbsp;·&nbsp;  [LightGBM and XGBoost, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/lightgbm-and-xgboost-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Missing Values](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/lightgbm-and-xgboost-deep-dive/08-missing-values.ipynb) &#8594;
